# **Phase 2 - Fine-tune Bi-Encoder**

In [1]:
%pip install -qU transformers datasets sentence-transformers accelerate pandas pyarrow numpy tqdm scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
%%writefile workspace/finetune_biencoder.py

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

from datasets                            import Dataset as HFDataset
from sentence_transformers               import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses        import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation    import InformationRetrievalEvaluator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


print('\nRead data:\n')
DATA_DIR  = Path('workspace/data/cleaned')
MODEL_DIR = Path('workspace/models/biencoder')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

corpus      = pd.read_parquet(DATA_DIR / 'corpus.parquet')
val_split   = pd.read_parquet(DATA_DIR / 'val_split.parquet')
train_split = pd.read_parquet(DATA_DIR / 'train_split.parquet')
cid2text    = {str(cid): text for cid, text in zip(corpus['cid'].tolist(), corpus['text'].tolist())}


print('\nPrepare evaluation data:\n')
val_sample       = val_split.sample(n=1000, random_state=42)
queries_dict     = {str(row.qid): row.question                 for _, row in val_sample.iterrows()}
relevant_dict    = {str(row.qid): set(str(c) for c in row.cid) for _, row in val_sample.iterrows()}

all_cids         = list(cid2text.keys())
val_pos_cids     = set().union(*relevant_dict.values())
remaining_cids   = [str(cid) for cid in all_cids if str(cid) not in val_pos_cids]
val_neg_cids     = np.random.choice(remaining_cids, size=min(max(0, 5000 - len(val_pos_cids)), len(remaining_cids)), replace=False)

eval_cids        = val_pos_cids.union(set(val_neg_cids))
eval_corpus_dict = {cid: cid2text[cid] for cid in eval_cids if cid in cid2text}

print(f"- Original corpus size: {len(cid2text):,}")
print(f"- Eval corpus size    : {len(eval_corpus_dict):,} (pos={len(val_pos_cids):,}, neg={len(val_neg_cids):,})")
print(f"- Queries size        : {len(queries_dict):,}")


ir_evaluator = InformationRetrievalEvaluator(
    queries           = queries_dict,
    corpus            = eval_corpus_dict,
    relevant_docs     = relevant_dict,
    mrr_at_k          = [1, 3, 5, 10],
    ndcg_at_k         = [1, 3, 5, 10],
    accuracy_at_k     = [1, 3, 5, 10],
    name              = 'val_ir',
    batch_size        = 512,
    show_progress_bar = True
)


print('\nPrepare training data:\n')
def load_training_data():
    anchors, positives = [], []
    for _, row in train_split.iterrows():
        for c in row['context']:
            anchors.append(str(row['question']).strip())
            positives.append(str(c).strip())
    print(f"- Total pairs: {len(anchors):,}")
    return HFDataset.from_dict({'anchor': anchors, 'positive': positives})
train_dataset = load_training_data()


if __name__ == '__main__':
    model = SentenceTransformer('AITeamVN/Vietnamese_Embedding', model_kwargs={'attn_implementation': 'sdpa'})
    model.max_seq_length = 512

    args = SentenceTransformerTrainingArguments(
        output_dir                    = str(MODEL_DIR / 'checkpoints'),
        num_train_epochs              = 1,
        per_device_train_batch_size   = 64,
        gradient_accumulation_steps   = 1,
        learning_rate                 = 1e-5,
        warmup_steps                  = 100,
        weight_decay                  = 0.01,
        lr_scheduler_type             = 'cosine_with_restarts',
        eval_strategy                 = 'steps',
        eval_steps                    = 500,
        save_strategy                 = 'steps',
        save_steps                    = 1000,
        save_total_limit              = 1,
        load_best_model_at_end        = True,
        metric_for_best_model         = 'val_ir_cosine_ndcg@10',
        greater_is_better             = True,
        logging_steps                 = 50,
        fp16                          = False,
        bf16                          = True,
        dataloader_num_workers        = 16,
        dataloader_pin_memory         = True,
        dataloader_persistent_workers = True,
        batch_sampler                 = BatchSamplers.NO_DUPLICATES,
        report_to                     = 'none',
        run_name                      = 'fine-tune-biencoder',
        gradient_checkpointing        = False,
    )
    loss = CachedMultipleNegativesRankingLoss(
        model           = model, 
        mini_batch_size = 64
    )
    trainer = SentenceTransformerTrainer(
        model         = model, 
        args          = args, 
        train_dataset = train_dataset, 
        loss          = loss, 
        evaluator     = ir_evaluator
    )

    print('\nBaseline evaluation:\n')
    baseline_metrics = trainer.evaluate()
    for key, value in baseline_metrics.items():
        if 'ndcg' in key or 'mrr' in key or 'accuracy' in key:
            print(f"\t- {key}: {value:.4f}")

    print('\nStart training:\n')
    trainer.train()

    print('\nSave model:\n')
    model.save_pretrained(str(MODEL_DIR / 'final_model'))

Overwriting workspace/finetune_biencoder.py


In [8]:
!accelerate launch --num_processes 1 --mixed_precision bf16 workspace/finetune_biencoder.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
//workspace/finetune_biencoder.py:17: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses        import CachedMultipleNegativesRankingLoss
//workspace/finetune_biencoder.py:18: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers
//workspace/finetune_biencoder.py:19: DeprecationWarning: Importing from 'sentence_tran

In [9]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path    = 'workspace/models/biencoder/final_model',
    repo_id        = 'YuITC/vietnamese-embedding-vn-legal',
    repo_type      = 'model',
    commit_message = 'update: retrain with batch_size=72, improved ndcg@10'
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/YuITC/vietnamese-embedding-vn-legal/commit/1877cdfb54d8381fb1ac3e2a66ad4f7f8a29caea', commit_message='update: retrain with batch_size=72, improved ndcg@10', commit_description='', oid='1877cdfb54d8381fb1ac3e2a66ad4f7f8a29caea', pr_url=None, repo_url=RepoUrl('https://huggingface.co/YuITC/vietnamese-embedding-vn-legal', endpoint='https://huggingface.co', repo_type='model', repo_id='YuITC/vietnamese-embedding-vn-legal'), pr_revision=None, pr_num=None)